%% [markdown]
# Evo2 inference parity — CPU / CUDA / Trainium  (DEV mode, FFT→conv1d)
Runs the **port** (vendored Evo2 with the inline FFT→conv1d Neuron patch) on every device and compares
to the **original FFT model** (ground-truth oracle, CPU). This proves BOTH: (a) the conv1d rewrite equals
the FFT original, and (b) CPU/Trainium device parity. `cuda` auto-skips when absent.
`NEURON_RT_VISIBLE_CORES=0 jupyter nbconvert --to notebook --execute 01_inference_parity.ipynb`.

In [1]:
# %%
import os
os.environ.setdefault("NEURON_RT_VISIBLE_CORES", "0")
import sys
sys.path.insert(0, os.path.abspath("../src"))
import torch
import torch.nn.functional as F
import evo2_reference as R

[W708 23:47:56.776874684 OperatorEntry.cpp:208] Warning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: aten::gather.out(Tensor self, int dim, Tensor index, *, bool sparse_grad=False, Tensor(a!) out) -> Tensor(a!)
    registered at /pytorch/build/aten/src/ATen/RegisterSchema.cpp:6
  dispatch key: PrivateUse1
  previous kernel: registered at /pytorch/build/aten/src/ATen/RegisterCPU_3.cpp:7637
       new kernel: registered at NeuronDispatcher.cpp:0 (function operator())


In [2]:
def devices():
    devs = ["cpu"]
    if torch.cuda.is_available():
        devs.append("cuda")
    try:
        import torch_neuronx  # noqa: F401
        devs.append("neuron")
    except Exception as e:
        print("neuron unavailable:", e)
    return devs

In [3]:
DEVICES = devices()
print("torch", torch.__version__, "| devices:", DEVICES, "| model:", R.MODEL_ID)

torch 2.11.0+cpu | devices: ['cpu', 'neuron'] | model: Taykhoom/Evo2-1B-8K


In [4]:
# %% [markdown]
# ## Ground-truth: original FFT model on CPU
# %%
ids, = R.build_inputs()
with torch.no_grad():
    ref = tuple(t.float() for t in R.load_fft_reference()(ids))   # (hidden, logits), FFT
for name, t in zip(R.OUTPUT_ORDER, ref):
    print(f"FFT-ref {name:7s} shape={tuple(t.shape)}")

/home/user/miniconda3/envs/torch-neuron/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Fetching 16 files: 100%|██████████| 16/16 [00:00<00:00, 169466.83it/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Fetching 16 files: 100%|██████████| 16/16 [00:00<00:00, 213722.50it/s]

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 269/269 [00:00<00:00, 10245.99it/s]

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


FFT-ref hidden  shape=(1, 32, 1920)
FFT-ref logits  shape=(1, 32, 512)


In [5]:
# %% [markdown]
# ## Run the port (conv1d) on each device; compare to the FFT reference
# %%
def run(device):
    model = R.load(device)                       # vendored conv1d port
    with torch.no_grad():
        out = model(ids.to(device))
    if device == "neuron":
        import torch_neuronx; torch_neuronx.synchronize()
    return tuple(t.detach().float().cpu() for t in out)

In [6]:
def cos(a, b): return F.cosine_similarity(a.flatten(), b.flatten(), dim=0).item()

In [7]:
all_ok = True
for d in DEVICES:
    out = run(d)
    print(f"\nport@{d} vs FFT reference:")
    for name, r, o in zip(R.OUTPUT_ORDER, ref, out):
        c = cos(r, o); ok = c >= 0.99
        all_ok = all_ok and ok
        print(f"  {name:7s} cosine={c:.6f}  max-abs={(r-o).abs().max():.3e}  {'OK' if ok else 'FAIL'}")
    t1 = (ref[1].argmax(-1) == out[1].argmax(-1)).float().mean().item() * 100
    print(f"  top-1 next-byte agreement={t1:.1f}%")

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Fetching 16 files: 100%|██████████| 16/16 [00:00<00:00, 174308.74it/s]

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 269/269 [00:00<00:00, 9798.84it/s]


port@cpu vs FFT reference:
  hidden  cosine=1.000001  max-abs=4.679e-06  OK
  logits  cosine=1.000000  max-abs=2.670e-05  OK
  top-1 next-byte agreement=100.0%


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Fetching 16 files: 100%|██████████| 16/16 [00:00<00:00, 146846.53it/s]

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 269/269 [00:00<00:00, 9967.21it/s]


port@neuron vs FFT reference:
  hidden  cosine=1.000001  max-abs=1.585e-02  OK
  logits  cosine=1.000000  max-abs=1.019e-01  OK
  top-1 next-byte agreement=100.0%


In [8]:
print("\nINFERENCE PARITY (rewrite + device):", "PASS" if all_ok else "FAIL")
assert all_ok, "port diverged from the FFT reference"


INFERENCE PARITY (rewrite + device): PASS
